# Kiểm tra bằng thuật toán tham chiếu: SPMF và FIMI

Chạy FPClose của hai cài đặt có sẵn trên 9 dataset benchmark, với 56 cấu hình minsup của
`data/results/benchmark_config.csv`:

- **SPMF FPClose** (Java, gọi qua `spmf.jar`): số closed itemsets phải bằng số tham chiếu
  `reference_closed_itemsets` (kết quả SPMF đã ghi khi chọn các mức minsup); mỗi dòng đúng định dạng,
  không có item trùng, support ≥ `minsup_count`.
- **FIMI FPclose** (C++ gốc của Grahne & Zhu, FIMI'04): tập closed itemsets và support phải
  giống SPMF **từng dòng**.

Phần này kiểm tra *dữ liệu* và *công cụ tham chiếu*, chưa kiểm tra implementation của nhóm.
Thời gian đo ở đây gồm cả khởi động JVM, không dùng làm số benchmark.

**Cách chạy:** mở notebook với kernel Julia (VS Code, Jupyter hoặc Colab với runtime Julia) rồi *Run all*,
hoặc chạy lần lượt từng cell. Notebook tự chuẩn bị những gì còn thiếu:
- 9 dataset: dùng `data/benchmark/` nếu đã chạy `data_setup`, nếu không thì tải bản công khai trên
  Hugging Face và kiểm tra SHA-256;
- Java 25 (trên Linux/Colab tự tải Temurin 25), `spmf.jar`, và biên dịch FPclose của FIMI bằng `g++`.

Toàn bộ mất khoảng 10 phút.

## 0 · Thiết lập thư mục và cấu hình chạy

Xác định nơi đặt dữ liệu, công cụ và kết quả; ghi ra bảng 56 cấu hình minsup và SHA-256 của 9 dataset.

In [ ]:
# Thư mục gốc: repo FPClose nếu đang ở trong repo, ngược lại là ./fim_data (giống data_setup)
function find_repo(dir = pwd())
    while true
        isfile(joinpath(dir, "data", "benchmark", "MANIFEST.md")) && return dir
        parent = dirname(dir)
        parent == dir && return nothing
        dir = parent
    end
end
REPO = find_repo()
ROOT = REPO === nothing ? joinpath(pwd(), "fim_data") : REPO
BENCHMARK_DIR = mkpath(joinpath(ROOT, "data", "benchmark"))
TOOLS_DIR = mkpath(joinpath(ROOT, "tools"))
OUT_DIR = mkpath(joinpath(ROOT, "out"))

# 56 cấu hình minsup và số closed itemsets tham chiếu (data/results/benchmark_config.csv)
CONFIG_DIR = mkpath(joinpath(tempdir(), "fim_reference_check"))
CONFIG_CSV = joinpath(CONFIG_DIR, "benchmark_config.csv")
write(CONFIG_CSV, raw"""
dataset,n_transactions,minsup_pct,minsup_count,reference_closed_itemsets,pilot_wall_time_s,pilot_output_mb,reference_algorithm,status
chess,3196,90.0,2877,498,1.216,0.011,SPMF FPClose,PASS
chess,3196,85.0,2717,1885,0.786,0.045,SPMF FPClose,PASS
chess,3196,80.0,2557,5083,0.82,0.13,SPMF FPClose,PASS
chess,3196,75.0,2397,11525,1.379,0.313,SPMF FPClose,PASS
chess,3196,70.0,2238,23892,2.461,0.679,SPMF FPClose,PASS
chess,3196,65.0,2078,49034,7.988,1.452,SPMF FPClose,PASS
chess,3196,60.0,1918,98392,43.773,3.037,SPMF FPClose,PASS
mushrooms,8124,20.0,1625,1197,0.385,0.033,SPMF FPClose,PASS
mushrooms,8124,15.0,1219,2261,0.365,0.065,SPMF FPClose,PASS
mushrooms,8124,10.0,813,4885,0.386,0.148,SPMF FPClose,PASS
mushrooms,8124,7.0,569,8212,0.465,0.256,SPMF FPClose,PASS
mushrooms,8124,5.0,407,12843,0.532,0.416,SPMF FPClose,PASS
mushrooms,8124,3.0,244,22229,0.819,0.757,SPMF FPClose,PASS
mushrooms,8124,2.0,163,31654,1.126,1.124,SPMF FPClose,PASS
retail,88162,1.0,882,159,2.113,0.003,SPMF FPClose,PASS
retail,88162,0.5,441,580,2.245,0.009,SPMF FPClose,PASS
retail,88162,0.2,177,2691,2.246,0.044,SPMF FPClose,PASS
retail,88162,0.1,89,7572,2.37,0.128,SPMF FPClose,PASS
retail,88162,0.05,45,19114,3.381,0.327,SPMF FPClose,PASS
retail,88162,0.02,18,65301,5.201,1.184,SPMF FPClose,PASS
T10I4D100K,100000,1.0,1000,385,3.041,0.006,SPMF FPClose,PASS
T10I4D100K,100000,0.5,500,1073,2.763,0.017,SPMF FPClose,PASS
T10I4D100K,100000,0.2,200,13107,4.597,0.286,SPMF FPClose,PASS
T10I4D100K,100000,0.1,100,26806,3.315,0.581,SPMF FPClose,PASS
T10I4D100K,100000,0.05,50,46993,3.725,0.981,SPMF FPClose,PASS
T10I4D100K,100000,0.02,20,107822,5.199,2.079,SPMF FPClose,PASS
accidents,340183,90.0,306165,31,2.764,0.001,SPMF FPClose,PASS
accidents,340183,80.0,272147,149,2.814,0.003,SPMF FPClose,PASS
accidents,340183,70.0,238129,529,3.2,0.012,SPMF FPClose,PASS
accidents,340183,60.0,204110,2074,4.143,0.051,SPMF FPClose,PASS
accidents,340183,50.0,170092,8057,4.854,0.214,SPMF FPClose,PASS
accidents,340183,40.0,136074,32528,7.003,0.93,SPMF FPClose,PASS
connect,67557,95.0,64180,811,2.74,0.022,SPMF FPClose,PASS
connect,67557,90.0,60802,3486,2.321,0.118,SPMF FPClose,PASS
connect,67557,85.0,57424,8252,1.643,0.31,SPMF FPClose,PASS
connect,67557,80.0,54046,15107,3.016,0.619,SPMF FPClose,PASS
connect,67557,75.0,50668,24346,4.719,1.063,SPMF FPClose,PASS
connect,67557,70.0,47290,35875,6.73,1.654,SPMF FPClose,PASS
kosarak,990002,1.0,9901,383,5.108,0.007,SPMF FPClose,PASS
kosarak,990002,0.5,4951,1618,5.549,0.031,SPMF FPClose,PASS
kosarak,990002,0.4,3961,2521,8.736,0.048,SPMF FPClose,PASS
kosarak,990002,0.3,2971,4982,9.394,0.099,SPMF FPClose,PASS
kosarak,990002,0.25,2476,8770,12.413,0.19,SPMF FPClose,PASS
kosarak,990002,0.2,1981,35864,18.435,1.054,SPMF FPClose,PASS
pumsb_star,49046,70.0,34333,17,4.297,0.0,SPMF FPClose,PASS
pumsb_star,49046,60.0,29428,68,1.497,0.002,SPMF FPClose,PASS
pumsb_star,49046,50.0,24523,248,2.893,0.007,SPMF FPClose,PASS
pumsb_star,49046,45.0,22071,713,2.006,0.021,SPMF FPClose,PASS
pumsb_star,49046,40.0,19619,2610,2.284,0.098,SPMF FPClose,PASS
pumsb_star,49046,35.0,17167,6129,3.825,0.253,SPMF FPClose,PASS
T40I10D100K,100000,5.0,5000,316,15.477,0.005,SPMF FPClose,PASS
T40I10D100K,100000,3.0,3000,793,22.597,0.013,SPMF FPClose,PASS
T40I10D100K,100000,2.0,2000,2293,20.97,0.039,SPMF FPClose,PASS
T40I10D100K,100000,1.5,1500,6539,24.214,0.13,SPMF FPClose,PASS
T40I10D100K,100000,1.25,1250,13460,27.445,0.295,SPMF FPClose,PASS
T40I10D100K,100000,1.0,1000,65236,35.049,1.86,SPMF FPClose,PASS
""")
# SHA-256 và số giao dịch của 9 dataset (data/results/benchmark_inventory.csv)
INVENTORY_CSV = joinpath(CONFIG_DIR, "benchmark_inventory.csv")
write(INVENTORY_CSV, raw"""
dataset,exists,transactions,expected,size_mb,sha256,status
T10I4D100K,True,100000,100000,3.741,fb2dd118ff822ff88d724e47d5a4cea6531c0bd6294397db4d36890c1450f1a7,PASS
T40I10D100K,True,100000,100000,14.668,e3d86e8cf6eec0093528e238ab8a4ba10471375c00a9298144edb462caadb28c,PASS
accidents,True,340183,340183,33.54,c2b3762e3f8f0ca241c35359b24e3015fd34ea241d09f94b42ae44ed42914726,PASS
chess,True,3196,3196,0.323,341d0ff8676e053dd3b417d61f84c732736154f987ce56975a099b07460957f5,PASS
connect,True,67557,67557,8.762,650a92d7803c465fb08fc0e58de087ceab84cd5f0de4c6816b523774dcd6a2c9,PASS
kosarak,True,990002,990002,30.546,cb4270629d786207143bb3c4002c36aedc3824f55294c7b3d5538c8023cec314,PASS
mushrooms,True,8124,8124,0.536,06206c73983251a9488414c4379a030e836bbeb6fe9eff1b5624a0193a811640,PASS
pumsb_star,True,49046,49046,10.724,a9ecd5c882ffff9e0042703253d0eef3453f747537e0027c9be275a8aaf38f2f,PASS
retail,True,88162,88162,3.892,417563fb5feb3711d4f761230ca78b76d100fe2ee0d3178fcc4fbb000d8d1c36,PASS
""")

DATASETS = nothing   # vd. ["chess", "mushrooms"] để chạy thử nhanh; nothing = cả 9 dataset
TIMEOUT = 600        # giây cho mỗi lần chạy

println("Julia ", VERSION)
println("Thư mục làm việc: ", ROOT)

### Hàm dùng chung: tính SHA-256, đọc và ghi CSV

Dùng ở mọi bước bên dưới để kiểm tra file sinh ra có đúng từng byte với bản nhóm đã công bố hay không.

Hàm dùng chung cho các script chuẩn bị dữ liệu benchmark.
Chỉ dùng thư viện chuẩn của Julia (SHA), không cần thêm gói nào.

In [ ]:
using SHA

"""
    sha256_file(path) -> String

SHA-256 (hex, chữ thường) của file, đọc theo luồng nên không tốn RAM với file lớn.
"""
sha256_file(path::AbstractString) = open(io -> bytes2hex(sha256(io)), path)

"""
    read_csv_rows(path) -> Vector{Dict{String,String}}

Đọc CSV đơn giản (không có dấu phẩy trong ô), đủ cho các file trong `data/results/`.
"""
function read_csv_rows(path::AbstractString)
    lines = filter(!isempty, strip.(readlines(path)))
    header = split(lines[1], ',')
    return [Dict(String(h) => String(v) for (h, v) in zip(header, split(l, ','))) for l in lines[2:end]]
end

"""
    write_csv(path, header, rows)

Ghi CSV với xuống dòng LF. `rows` là danh sách vector giá trị theo đúng thứ tự `header`.
"""
function write_csv(path::AbstractString, header::Vector{String}, rows)
    mkpath(dirname(abspath(path)))
    open(path, "w") do io
        println(io, join(header, ','))
        for r in rows
            println(io, join(string.(r), ','))
        end
    end
end

## 1 · Chuẩn bị dữ liệu và công cụ

### Tải 9 dataset từ Hugging Face (nếu chưa có) và kiểm tra SHA-256

Tải 9 dataset công khai từ Hugging Face và so SHA-256, số giao dịch với
data/results/benchmark_inventory.csv. Không cần token (repo public).

File đã có và đúng hash thì không tải lại.

In [ ]:
using Downloads

const HF_REPO = "hytrang/fpclose-fim-benchmarks"
hf_url(name) = "https://huggingface.co/datasets/$HF_REPO/resolve/main/$name"

"""
    verify_hf(inventory, outdir; names = nothing) -> Bool

Trả về `true` nếu mọi file khớp hash và số giao dịch. `names` giới hạn danh sách dataset.
"""
function verify_hf(inventory::AbstractString, outdir::AbstractString; names = nothing)
    mkpath(outdir)
    ok = true
    for row in read_csv_rows(inventory)
        name = row["dataset"]
        names === nothing || name in names || continue
        file = joinpath(outdir, "$name.txt")
        expected = lowercase(row["sha256"])
        if !(isfile(file) && sha256_file(file) == expected)
            Downloads.download(hf_url("$name.txt"), file)
        end
        actual = sha256_file(file)
        lines = countlines(file)
        good = actual == expected && lines == parse(Int, row["transactions"])
        ok &= good
        println(good ? "[PASS] " : "[FAIL] ", rpad(name, 12), lines, " giao dịch  ", actual)
    end
    return ok
end

### Tìm hoặc cài Java 25, tải spmf.jar và biên dịch FIMI FPclose

Chuẩn bị các công cụ tham chiếu cho bước 2 và bước 3:
- Java 25 (spmf.jar mới được build cho Java 25),
- tools/spmf.jar,
- tools/fimi/fim_closed: FPclose gốc của Grahne & Zhu (FIMI'04), biên dịch từ fimi06b.tgz.

Thứ nào đã có thì bỏ qua. Trên Linux (Colab) tự tải Temurin 25; trên Windows cần cài sẵn
Temurin 25 và g++ (vd. Strawberry Perl / MSYS2).

In [ ]:
using Downloads

"Tìm java.exe của JDK 25 (Temurin) nếu `java` trên PATH là bản cũ."
function find_java()
    for root in ("C:\\Program Files\\Eclipse Adoptium", "/usr/lib/jvm")
        isdir(root) || continue
        for d in sort(readdir(root); rev = true)
            startswith(d, "jdk-25") || startswith(d, "temurin-25") || continue
            for exe in ("java.exe", "java")
                p = joinpath(root, d, "bin", exe)
                isfile(p) && return p
            end
        end
    end
    return "java"
end


const SPMF_JAR_URL = "https://www.philippe-fournier-viger.com/spmf/spmf.jar"
const FIMI_SRC_URL = "https://fimi.uantwerpen.be/src/fimi06b.tgz"
const TEMURIN25_LINUX_URL = "https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse"

"Phiên bản chính của `java` (vd. 25), 0 nếu không chạy được."
function java_major(java)
    buf = IOBuffer()
    try
        run(pipeline(ignorestatus(`$java -version`); stdout = buf, stderr = buf))
    catch
        return 0
    end
    m = match(r"version \"(\d+)", String(take!(buf)))
    return m === nothing ? 0 : parse(Int, m[1])
end

"Tìm hoặc tải Java ≥ 25; trả về đường dẫn tới lệnh java."
function ensure_java(toolsdir)
    candidates = [find_java()]
    if isdir(toolsdir)
        for d in readdir(toolsdir; join = true)
            startswith(basename(d), "jdk-25") && push!(candidates, joinpath(d, "bin", Sys.iswindows() ? "java.exe" : "java"))
        end
    end
    for java in candidates
        java_major(java) >= 25 && return java
    end
    Sys.islinux() || error("Cần Java 25: cài Eclipse Temurin 25 (https://adoptium.net) rồi chạy lại")
    mkpath(toolsdir)
    tarball = joinpath(toolsdir, "temurin25.tar.gz")
    println("Tải Temurin 25 (~200 MB) ...")
    Downloads.download(TEMURIN25_LINUX_URL, tarball)
    run(Cmd(`tar -xzf temurin25.tar.gz`; dir = toolsdir))
    rm(tarball)
    for d in readdir(toolsdir; join = true)
        java = joinpath(d, "bin", "java")
        startswith(basename(d), "jdk-25") && java_major(java) >= 25 && return java
    end
    error("Đã tải Temurin 25 nhưng không tìm thấy bin/java trong $toolsdir")
end

"Tải spmf.jar nếu chưa có."
function ensure_spmf(toolsdir)
    jar = joinpath(toolsdir, "spmf.jar")
    if !isfile(jar)
        mkpath(toolsdir)
        println("Tải spmf.jar ...")
        Downloads.download(SPMF_JAR_URL, jar)
    end
    return jar
end

"Tải fimi06b.tgz và biên dịch FPclose (`-DCFI`) nếu chưa có."
function ensure_fimi(toolsdir)
    fimidir = mkpath(joinpath(toolsdir, "fimi"))
    exe = joinpath(fimidir, Sys.iswindows() ? "fim_closed.exe" : "fim_closed")
    isfile(exe) && return exe
    src = mkpath(joinpath(fimidir, "src"))
    println("Tải và biên dịch FIMI FPclose ...")
    Downloads.download(FIMI_SRC_URL, joinpath(src, "fimi06b.tgz"))
    run(Cmd(`tar -xzf fimi06b.tgz`; dir = src))
    # Code năm 2003 dùng các header cũ (<iostream.h>, ...); thêm header tương thích,
    # không sửa code gốc.
    compat = mkpath(joinpath(src, "compat"))
    for (old, new) in ("iostream.h" => "iostream", "fstream.h" => "fstream",
                       "iomanip.h" => "iomanip", "strstream.h" => "sstream")
        write(joinpath(compat, old), "#pragma once\n#include <$new>\nusing namespace std;\n")
    end
    sources = ["fpmax.cpp", "data.cpp", "fp_tree.cpp", "buffer.cpp", "fsout.cpp", "fp_node.cpp"]
    run(Cmd(`g++ -O3 -w -fpermissive -I compat -DCFI $sources -o $exe`; dir = src))
    return exe
end

"""
    setup_tools(toolsdir) -> (java, jar, fimi)
"""
function setup_tools(toolsdir)
    java = ensure_java(toolsdir)
    jar = ensure_spmf(toolsdir)
    fimi = ensure_fimi(toolsdir)
    println("java  ", java, "  (Java ", java_major(java), ")")
    println("spmf  ", jar)
    println("fimi  ", fimi)
    return (java = java, jar = jar, fimi = fimi)
end

### Chạy bước 1

File đã có và đúng SHA-256 thì không tải lại; công cụ đã có thì không cài lại.

In [ ]:
@assert verify_hf(INVENTORY_CSV, BENCHMARK_DIR; names = DATASETS) "Có dataset không khớp SHA-256"
TOOLS = setup_tools(TOOLS_DIR)

## 2 · SPMF FPClose so với số tham chiếu

### Chạy SPMF FPClose trên từng cấu hình, kiểm tra output và đếm closed itemsets

Chạy SPMF FPClose trên các cấu hình trong benchmark_config.csv và so số closed
itemsets với kết quả pilot đã ghi (cột reference_closed_itemsets).

Cần: Java 25 và tools/spmf.jar. Mỗi output của SPMF được kiểm:
- mỗi dòng có dạng `i1 i2 ... #SUP: s`, item không trùng,
- support ≥ minsup_count (= ceil(n × minsup% / 100)),
- số dòng = reference_closed_itemsets của pilot.

Đây chỉ là kiểm tra dữ liệu + SPMF; không đánh giá implementation của nhóm.

In [9]:
"""
    check_spmf_output(path, minsup_count) -> Int

Đếm closed itemsets trong output SPMF và kiểm tra định dạng, support.
"""
function check_spmf_output(path::AbstractString, minsup_count::Integer)
    count = 0
    for line in eachline(path)
        s = strip(line)
        (isempty(s) || startswith(s, '@') || startswith(s, '#')) && continue
        parts = split(s, "#SUP:")
        length(parts) == 2 || error("Dòng output lạ: $(first(s, 120))")
        items = parse.(Int, split(parts[1]))
        support = parse(Int, strip(parts[2]))
        allunique(items) || error("Itemset có item trùng: $(first(s, 120))")
        support >= minsup_count || error("Support $support < minsup_count $minsup_count")
        count += 1
    end
    return count
end

"Chạy SPMF FPClose một lần; trả về (status, wall_time_s, n_closed, output_mb)."
function run_spmf(java, jar, input, minsup_pct, minsup_count; timeout_s = 300)
    out = tempname() * "_closed.txt"
    log = tempname() * "_spmf.log"
    pct = string(minsup_pct)
    pct = endswith(pct, ".0") ? pct[1:end-2] : pct
    cmd = `$java -Xmx4g -jar $jar run FPClose $input $out $(pct)%`
    t0 = time()
    proc = open(log, "w") do io
        run(pipeline(cmd; stdout = io, stderr = io); wait = false)
    end
    while process_running(proc)
        if time() - t0 > timeout_s
            kill(proc)
            wait(proc)
            rm(out; force = true)
            return (status = "TIMEOUT", wall_time_s = round(time() - t0; digits = 3),
                n_closed = missing, output_mb = missing)
        end
        sleep(0.05)
    end
    elapsed = round(time() - t0; digits = 3)
    try
        if !success(proc) || !isfile(out)
            msg = isfile(log) ? strip(read(log, String)) : ""
            return (status = "FAILED: " * first(replace(msg, '\n' => ' '), 200),
                wall_time_s = elapsed, n_closed = missing, output_mb = missing)
        end
        n = check_spmf_output(out, minsup_count)
        return (status = "OK", wall_time_s = elapsed, n_closed = n,
            output_mb = round(filesize(out) / 2^20; digits = 3))
    finally
        rm(out; force = true)
        rm(log; force = true)
    end
end

"""
    spmf_check(config, datadir; java, jar, datasets, levels, timeout_s) -> Vector{NamedTuple}
"""
function spmf_check(config::AbstractString, datadir::AbstractString; java = find_java(),
        jar, datasets = nothing,
        levels = "all", timeout_s = 300)
    isfile(jar) || error("Không thấy $jar")
    rows = read_csv_rows(config)
    if levels == "top"
        best = Dict{String,Dict{String,String}}()
        for r in rows
            d = r["dataset"]
            if !haskey(best, d) || parse(Float64, r["minsup_pct"]) > parse(Float64, best[d]["minsup_pct"])
                best[d] = r
            end
        end
        rows = [r for r in rows if best[r["dataset"]] === r]
    end
    results = NamedTuple[]
    for r in rows
        d = r["dataset"]
        datasets === nothing || d in datasets || continue
        input = joinpath(datadir, "$d.txt")
        isfile(input) || error("Không thấy $input — chạy data_setup trước")
        pct = parse(Float64, r["minsup_pct"])
        mc = parse(Int, r["minsup_count"])
        ref = parse(Int, r["reference_closed_itemsets"])
        res = run_spmf(java, jar, input, pct, mc; timeout_s = timeout_s)
        match = res.status == "OK" && res.n_closed == ref
        push!(results, (dataset = d, minsup_pct = pct, minsup_count = mc,
            reference_closed_itemsets = ref, n_closed_itemsets = res.n_closed,
            match = match, wall_time_s = res.wall_time_s, output_mb = res.output_mb,
            status = res.status))
        println(match ? "[OK]   " : "[SAI]  ", rpad(d, 12), lpad(string(pct, "%"), 7),
            "  ref=", lpad(ref, 9), "  spmf=", lpad(coalesce(res.n_closed, "-"), 9),
            "  ", res.wall_time_s, "s  ", res.status == "OK" ? "" : res.status)
    end
    return results
end

const SPMF_CHECK_HEADER = ["dataset", "minsup_pct", "minsup_count", "reference_closed_itemsets",
    "n_closed_itemsets", "match", "wall_time_s", "output_mb", "status"]

9-element Vector{String}:
 "dataset"
 "minsup_pct"
 "minsup_count"
 "reference_closed_itemsets"
 "n_closed_itemsets"
 "match"
 "wall_time_s"
 "output_mb"
 "status"

### Chạy bước 2

In [10]:
spmf_results = spmf_check(CONFIG_CSV, BENCHMARK_DIR; java = TOOLS.java, jar = TOOLS.jar,
    datasets = DATASETS, timeout_s = TIMEOUT)
write_csv(joinpath(OUT_DIR, "spmf_check.csv"), SPMF_CHECK_HEADER,
    [[coalesce(getfield(r, Symbol(h)), "") for h in SPMF_CHECK_HEADER] for r in spmf_results])
n_ok = count(r -> r.match, spmf_results)
println("\n", n_ok, "/", length(spmf_results), " cấu hình khớp số tham chiếu")
@assert n_ok == length(spmf_results)

[OK]   chess         90.0%  ref=      498  spmf=      498  0.374s  
[OK]   chess         85.0%  ref=     1885  spmf=     1885  0.375s  
[OK]   chess         80.0%  ref=     5083  spmf=     5083  0.496s  
[OK]   chess         75.0%  ref=    11525  spmf=    11525  0.694s  
[OK]   chess         70.0%  ref=    23892  spmf=    23892  1.187s  
[OK]   chess         65.0%  ref=    49034  spmf=    49034  3.498s  
[OK]   chess         60.0%  ref=    98392  spmf=    98392  19.801s  
[OK]   mushrooms     20.0%  ref=     1169  spmf=     1169  0.442s  
[OK]   mushrooms     15.0%  ref=     2209  spmf=     2209  0.446s  
[OK]   mushrooms     10.0%  ref=     4850  spmf=     4850  0.636s  
[OK]   mushrooms      7.0%  ref=     8202  spmf=     8202  0.75s  
[OK]   mushrooms      5.0%  ref=    12789  spmf=    12789  0.703s  
[OK]   mushrooms      3.0%  ref=    22380  spmf=    22380  0.951s  
[OK]   mushrooms      2.0%  ref=    31416  spmf=    31416  1.428s  
[OK]   retail         1.0%  ref=      159  spmf=

## 3 · FIMI FPclose so với SPMF, từng itemset

### Chạy cả FIMI và SPMF trên từng cấu hình rồi so hai tập itemset + support

So FPclose gốc của Grahne & Zhu (kho FIMI) với SPMF FPClose trên từng cấu hình của
benchmark_config.csv: cùng tập closed itemsets và cùng support, không chỉ cùng số lượng.

Cần: tools/spmf.jar + Java 25, và tools/fimi/fim_closed(.exe) biên dịch từ
https://fimi.uantwerpen.be/src/fimi06b.tgz. Khác biệt định dạng:
- FIMI ghi `1 3 (3)`, SPMF ghi `1 3 #SUP: 3`;
- FIMI in thêm tập rỗng `(n)` khi không item nào có mặt trong mọi giao dịch;
  SPMF không in tập rỗng, nên tập rỗng được bỏ trước khi so.

In [11]:
"Chạy lệnh với giới hạn thời gian; trả về (status, wall_time_s)."
function run_timed(cmd::Cmd; timeout_s = 300)
    log = tempname() * ".log"
    t0 = time()
    proc = open(log, "w") do io
        run(pipeline(cmd; stdout = io, stderr = io); wait = false)
    end
    while process_running(proc)
        if time() - t0 > timeout_s
            kill(proc)
            wait(proc)
            rm(log; force = true)
            return ("TIMEOUT", round(time() - t0; digits = 3))
        end
        sleep(0.02)
    end
    elapsed = round(time() - t0; digits = 3)
    msg = success(proc) ? "OK" : "FAILED: " * first(replace(strip(read(log, String)), '\n' => ' '), 200)
    rm(log; force = true)
    return (msg, elapsed)
end

"Đọc output thành tập `\"i1 i2 ...#sup\"` (item đã sắp tăng), bỏ tập rỗng."
function read_itemsets(path::AbstractString, format::Symbol)
    result = Set{String}()
    for line in eachline(path)
        s = strip(line)
        (isempty(s) || startswith(s, '@')) && continue
        if format == :spmf
            lhs, rhs = split(s, "#SUP:")
        else
            m = match(r"^(.*)\((\d+)\)$", s)
            m === nothing && error("Dòng FIMI lạ: $(first(s, 120))")
            lhs, rhs = m.captures
        end
        items = sort!(parse.(Int, split(lhs)))
        isempty(items) && continue
        key = join(items, ' ') * "#" * strip(rhs)
        key in result && error("Itemset bị lặp trong output $format: $key")
        push!(result, key)
    end
    return result
end

"""
    fimi_compare(config, datadir; ...) -> Vector{NamedTuple}
"""
function fimi_compare(config::AbstractString, datadir::AbstractString; java = find_java(),
        jar, fimi,
        datasets = nothing, timeout_s = 300)
    isfile(jar) || error("Không thấy $jar")
    isfile(fimi) || error("Không thấy $fimi — biên dịch FPclose của FIMI trước")
    results = NamedTuple[]
    for r in read_csv_rows(config)
        d = r["dataset"]
        datasets === nothing || d in datasets || continue
        input = joinpath(datadir, "$d.txt")
        pct = parse(Float64, r["minsup_pct"])
        mc = parse(Int, r["minsup_count"])
        ref = parse(Int, r["reference_closed_itemsets"])
        pct_s = string(pct)
        pct_s = endswith(pct_s, ".0") ? pct_s[1:end-2] : pct_s
        spmf_out = tempname() * "_spmf.txt"
        fimi_out = tempname() * "_fimi.txt"
        try
            s_status, s_time = run_timed(`$java -Xmx4g -jar $jar run FPClose $input $spmf_out $(pct_s)%`; timeout_s)
            f_status, f_time = run_timed(`$fimi $input $mc $fimi_out`; timeout_s)
            spmf = s_status == "OK" ? read_itemsets(spmf_out, :spmf) : nothing
            fim = f_status == "OK" ? read_itemsets(fimi_out, :fimi) : nothing
            same = spmf !== nothing && fim !== nothing && spmf == fim
            only_spmf = spmf !== nothing && fim !== nothing ? length(setdiff(spmf, fim)) : missing
            only_fimi = spmf !== nothing && fim !== nothing ? length(setdiff(fim, spmf)) : missing
            n_s = spmf === nothing ? missing : length(spmf)
            n_f = fim === nothing ? missing : length(fim)
            ok = same && n_s == ref
            push!(results, (dataset = d, minsup_pct = pct, minsup_count = mc,
                reference_closed_itemsets = ref, n_spmf = n_s, n_fimi = n_f,
                only_spmf = only_spmf, only_fimi = only_fimi, identical = same,
                spmf_time_s = s_time, fimi_time_s = f_time,
                status = s_status == "OK" && f_status == "OK" ? "OK" : "spmf=$s_status; fimi=$f_status"))
            println(ok ? "[OK]   " : "[SAI]  ", rpad(d, 12), lpad(string(pct, "%"), 7),
                "  ref=", lpad(ref, 8), "  spmf=", lpad(coalesce(n_s, "-"), 8),
                "  fimi=", lpad(coalesce(n_f, "-"), 8), "  giống từng dòng=", same,
                "  spmf ", s_time, "s / fimi ", f_time, "s")
        finally
            rm(spmf_out; force = true)
            rm(fimi_out; force = true)
        end
    end
    return results
end

const FIMI_COMPARE_HEADER = ["dataset", "minsup_pct", "minsup_count", "reference_closed_itemsets",
    "n_spmf", "n_fimi", "only_spmf", "only_fimi", "identical", "spmf_time_s", "fimi_time_s", "status"]

12-element Vector{String}:
 "dataset"
 "minsup_pct"
 "minsup_count"
 "reference_closed_itemsets"
 "n_spmf"
 "n_fimi"
 "only_spmf"
 "only_fimi"
 "identical"
 "spmf_time_s"
 "fimi_time_s"
 "status"

### Chạy bước 3

In [12]:
fimi_results = fimi_compare(CONFIG_CSV, BENCHMARK_DIR; java = TOOLS.java, jar = TOOLS.jar,
    fimi = TOOLS.fimi, datasets = DATASETS, timeout_s = TIMEOUT)
write_csv(joinpath(OUT_DIR, "fimi_compare.csv"), FIMI_COMPARE_HEADER,
    [[coalesce(getfield(r, Symbol(h)), "") for h in FIMI_COMPARE_HEADER] for r in fimi_results])
n_ok = count(r -> r.identical && r.n_spmf == r.reference_closed_itemsets, fimi_results)
println("\n", n_ok, "/", length(fimi_results), " cấu hình: SPMF = FIMI từng itemset và support, đúng số tham chiếu")
@assert n_ok == length(fimi_results)

[OK]   chess         90.0%  ref=     498  spmf=     498  fimi=     498  giống từng dòng=true  spmf 0.387s / fimi 0.271s
[OK]   chess         85.0%  ref=    1885  spmf=    1885  fimi=    1885  giống từng dòng=true  spmf 0.404s / fimi 0.062s
[OK]   chess         80.0%  ref=    5083  spmf=    5083  fimi=    5083  giống từng dòng=true  spmf 0.498s / fimi 0.062s
[OK]   chess         75.0%  ref=   11525  spmf=   11525  fimi=   11525  giống từng dòng=true  spmf 0.619s / fimi 0.062s
[OK]   chess         70.0%  ref=   23892  spmf=   23892  fimi=   23892  giống từng dòng=true  spmf 1.269s / fimi 0.081s
[OK]   chess         65.0%  ref=   49034  spmf=   49034  fimi=   49034  giống từng dòng=true  spmf 3.919s / fimi 0.154s
[OK]   chess         60.0%  ref=   98392  spmf=   98392  fimi=   98392  giống từng dòng=true  spmf 15.314s / fimi 0.218s
[OK]   mushrooms     20.0%  ref=    1169  spmf=    1169  fimi=    1169  giống từng dòng=true  spmf 0.433s / fimi 0.067s
[OK]   mushrooms     15.0%  ref=    220

## 4 · Tóm tắt theo dataset

In [13]:
println(rpad("dataset", 13), lpad("cấu hình", 9), lpad("SPMF=tham chiếu", 17), lpad("SPMF=FIMI", 11))
for d in unique(r.dataset for r in fimi_results)
    s = filter(r -> r.dataset == d, spmf_results)
    f = filter(r -> r.dataset == d, fimi_results)
    println(rpad(d, 13), lpad(length(f), 9),
        lpad(string(count(r -> r.match, s), "/", length(s)), 17),
        lpad(string(count(r -> r.identical, f), "/", length(f)), 11))
end
println("\nChi tiết: ", joinpath(OUT_DIR, "spmf_check.csv"), " và ", joinpath(OUT_DIR, "fimi_compare.csv"))

dataset       cấu hình  SPMF=tham chiếu  SPMF=FIMI
chess                7              7/7        7/7
mushrooms            7              7/7        7/7
retail               6              6/6        6/6
T10I4D100K           6              6/6        6/6
accidents            6              6/6        6/6
connect              6              6/6        6/6
kosarak              6              6/6        6/6
pumsb_star           6              6/6        6/6
T40I10D100K          6              6/6        6/6

Chi tiết: d:\7st_semester\Data\FPClose\out\spmf_check.csv và d:\7st_semester\Data\FPClose\out\fimi_compare.csv
